In [1]:
# Setup colab
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted")
import os
os.chdir('/content/drive/MyDrive/reranking_rag_and_qw/notebook')
print(f"✅ Working directory: {os.getcwd()}")
print("Contents:", os.listdir("."))

Mounted at /content/drive
✅ Drive mounted
✅ Working directory: /content/drive/MyDrive/reranking_rag_and_qw/notebook
Contents: ['.gitkeep', '04_train.ipynb', '01_setup_baseline.ipynb', '03_reranking.ipynb', '02_query_writing.ipynb']


In [ ]:
# Install requirements
%pip install -r ../requirements.txt

In [2]:
%pip install -q "sentence-transformers>=3.0"
%pip install -q "jsonlines"
%pip install -q "rank_bm25"
%pip install -q "rouge_score"
%pip install -q "faiss-cpu"

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 73.7 MB/s eta 0:00:00


# 2. Query Rewriting
Triển khai Multi-query, HyDE, và xử lý code-switching

In [3]:
import sys, os
root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

In [4]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from src.rewriting.query_rewriter import QueryRewriter
from src.generation.llm_generator import LLMGenerator
from src.data.data_loader import DataLoader

In [6]:
# Initialize
llm = LLMGenerator(provider="huggingface", hf_model="Qwen/Qwen2.5-7B-Instruct")
rewriter = QueryRewriter(llm)
data_loader = DataLoader("../data")

Loading Qwen/Qwen2.5-7B-Instruct on cuda...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Model loaded


## Test Multi-Query Rewriting

In [7]:
queries = [
    "Bệnh đái tháo đường là gì?",
    "Machine learning là gì?",
    "Cách phòng chống cúm"
]

for query in queries:
    rewritten = rewriter.rewrite(query, method="multi-query-hyde")
    print(f"Original: {query}")
    print(f"Rewritten ({len(rewritten)} queries):")
    for i, q in enumerate(rewritten, 1):
        print(f"  {i}. {q}")
    print()

Original: Bệnh đái tháo đường là gì?
Rewritten (3 queries):
  1. Bệnh đái tháo đường là gì?
  2. triệu chứng của bệnh đái tháo đường
  3. nguyên nhân gây ra bệnh đái tháo đường

Original: Machine learning là gì?
Rewritten (4 queries):
  1. Machine learning là gì?
  2. học máy là gì
  3. công nghệ học máy là gì
  4. Máy học là gì?

Original: Cách phòng chống cúm
Rewritten (4 queries):
  1. Cách phòng chống cúm
  2. phòng ngừa cúm
  3. cách ngăn chặn lây nhiễm cúm
  4. Viết đoạn văn giả định dựa trên kiến thức của tôi về cách phòng chống cúm:

Để phòng chống cúm hiệu quả, người dân nên tiêm vaccine cúm hàng năm theo khuyến cáo của CDC. Ngoài ra, việc rửa tay thường xuyên với xà phòng, đeo khẩu trang khi cần thiết, tránh tiếp xúc gần với người bệnh cũng rất quan trọng. Một chế độ ăn uống lành mạnh, đầy đủ休息中
请继续您的问题或任务，如果您有任何关于预防流感的问题或需要更详细的建议，请告诉我。我可以提供一些基本的预防措施信息。例如：

为了有效预防流感，人们应该每年按照CDC的建议接种流感疫苗。此外，经常用肥皂和



## Test Code-Switching Detection

In [8]:
code_switch_queries = [
    "Machine learning là gì?",
    "Cách setup Docker trên Ubuntu",
    "API là gì trong lập trình",
    "Triệu chứng bệnh cúm"
]

for query in code_switch_queries:
    has_switch = rewriter.has_code_switching(query)
    print(f"{query}")
    print(f"  Code-switching detected: {has_switch}")
    if has_switch:
        variations = rewriter.code_switching_handle(query)
        print(f"  Variations: {variations}")
    print()

Machine learning là gì?
  Code-switching detected: True
  Variations: ['Máy học là gì']

Cách setup Docker trên Ubuntu
  Code-switching detected: True
  Variations: ['Cách cài đặt Docker trên Ubuntu']

API là gì trong lập trình
  Code-switching detected: True
  Variations: ['Giao diện lập trình ứng dụng trong lập trình là API.']

Triệu chứng bệnh cúm
  Code-switching detected: False



## Evaluate Query Rewriting on Dataset

In [ ]:
# Load test data
test_data = data_loader.load_dataset("vhealthqa", "test")[:20]

results = []
for sample in test_data:
    query = sample.get("query", "") # Changed 'question' to 'query'
    if not query:
        continue

    rewritten = rewriter.rewrite(query, method="multi-query-hyde")

    results.append({
        "original": query,
        "num_queries": len(rewritten),
        "rewritten": rewritten
    })

print(f"Processed {len(results)} queries")
# Add a check to prevent ZeroDivisionError if results is empty
if len(results) > 0:
    print(f"Average rewritten count: {sum(r['num_queries'] for r in results) / len(results):.1f}")
else:
    print("No queries processed to calculate average rewritten count.")

# Show examples
for i, result in enumerate(results[:3]):
    print(f"\nExample {i+1}:")
    print(f"  Original: {result['original']}")
    print(f"  Rewritten ({result['num_queries']}):")
    for q in result['rewritten']:
        print(f"    - {q}")